<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks%20/%2006_diavigil_dynamic_resource_allocation_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup & Loading Audit Data
Python**

In [1]:
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Load audit test predictions and patient records
audit_data = joblib.load(os.path.join(DATA_PROCESSED, 'fairness_audit_results.joblib'))
test_df = pd.read_parquet(os.path.join(DATA_PROCESSED, 'audit_test_cohort.parquet'))

y_test = audit_data['y_test']
test_preds = audit_data['test_preds']
test_cal_probs = audit_data['test_cal_probs']

# Assemble patient evaluation dataframe
eval_df = test_df[['encounter_id', 'patient_nbr', 'age', 'time_in_hospital', 'number_inpatient']].copy()
eval_df['calibrated_risk'] = test_cal_probs
eval_df['is_high_risk'] = test_preds
eval_df['true_readmitted'] = y_test

print(f"Loaded Audit Cohort: {len(eval_df):,} encounters")
print(f"High-Risk Encounters Flagged: {eval_df['is_high_risk'].sum():,} ({eval_df['is_high_risk'].mean()*100:.2f}%)")

Mounted at /content/drive
Loaded Audit Cohort: 19,875 encounters
High-Risk Encounters Flagged: 2,742 (13.80%)


**Cell 2: Stochastic Monte Carlo Simulation**

Model post-discharge nurse intervention efficacy and contact intensity stochastically:

- Intervention Efficacy ($E_i$): Sampled from a Beta distribution ($\alpha=3.5, \beta=6.5$, mean $\approx 35\%$).
- Required Nurse Contact Hours ($H_i$): Derived dynamically from encounter complexity (longer inpatient stays and higher prior admissions require more contact time).

In [2]:
np.random.seed(42)
n_encounters = len(eval_df)
n_simulations = 1000

# Dynamic Contact Hours: Base 1.5 hrs + scaling with hospital length of stay & prior admissions
eval_df['required_nurse_hours'] = np.clip(
    1.5 + (0.25 * eval_df['time_in_hospital']) + (0.5 * eval_df['number_inpatient']),
    a_min=1.5,
    a_max=8.0
)

# Run Monte Carlo trials across high-risk candidates
high_risk_cohort = eval_df[eval_df['is_high_risk'] == 1].copy()

# Sample efficacy distributions: Beta(3.5, 6.5) -> mean = 0.35, plausible clinical range ~0.15 - 0.60
simulated_efficacies = np.random.beta(a=3.5, b=6.5, size=(len(high_risk_cohort), n_simulations))

# Expected readmissions averted per patient across trials: P(Readmit) * Efficacy
simulated_averted_matrix = high_risk_cohort['calibrated_risk'].values[:, None] * simulated_efficacies

# Compute empirical 95% Confidence Intervals
mean_averted_per_patient = simulated_averted_matrix.mean(axis=1)
ci_lower = np.percentile(simulated_averted_matrix.sum(axis=0), 2.5)
ci_upper = np.percentile(simulated_averted_matrix.sum(axis=0), 97.5)
total_mean_averted = simulated_averted_matrix.sum(axis=0).mean()

high_risk_cohort['expected_averted'] = mean_averted_per_patient

print("=== MONTE CARLO STOCHASTIC SIMULATION (1,000 TRIALS) ===")
print(f"Total High-Risk Flagged Cohort: {len(high_risk_cohort):,} patients")
print(f"Mean Expected Readmissions Prevented: {total_mean_averted:.2f}")
print(f"95% Empirical Credible Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")

=== MONTE CARLO STOCHASTIC SIMULATION (1,000 TRIALS) ===
Total High-Risk Flagged Cohort: 2,742 patients
Mean Expected Readmissions Prevented: 231.55
95% Empirical Credible Interval: [228.06, 235.12]


**Cell 3: Capacity-Constrained Resource Allocation (Knapsack Optimization)**

Hospitals have finite weekly care management hours. Allocate nurse contact hours using an efficiency-ranked greedy knapsack algorithm:$$\text{Efficiency}_i = \frac{\mathbb{E}[\text{Readmission Averted}_i]}{\text{Required Nurse Contact Hours}_i}$$

In [3]:
def optimize_under_capacity(cohort_df, available_nurse_hours=1500):
    df = cohort_df.copy()

    # Calculate marginal benefit per nurse hour
    df['efficiency_ratio'] = df['expected_averted'] / df['required_nurse_hours']

    # Rank by descending efficiency
    df = df.sort_values(by='efficiency_ratio', ascending=False).reset_index(drop=True)

    # Cumulative resource consumption
    df['cumulative_hours'] = df['required_nurse_hours'].cumsum()
    df['assigned_intervention'] = df['cumulative_hours'] <= available_nurse_hours

    selected = df[df['assigned_intervention']].copy()
    unselected = df[~df['assigned_intervention']].copy()

    return {
        'budget_hours': available_nurse_hours,
        'allocated_patients': len(selected),
        'total_hours_used': selected['required_nurse_hours'].sum(),
        'expected_readmissions_averted': selected['expected_averted'].sum(),
        'actual_readmissions_in_selected': selected['true_readmitted'].sum(),
        'precision_in_allocated': selected['true_readmitted'].mean(),
        'selected_cohort': selected
    }

# Test across 3 operational staffing budgets
budgets = [500, 1500, 3000]
allocation_results = [optimize_under_capacity(high_risk_cohort, b) for b in budgets]

alloc_summary = pd.DataFrame([{
    'Nurse Hour Budget': r['budget_hours'],
    'Patients Reached': r['allocated_patients'],
    'Hours Utilized': round(r['total_hours_used'], 1),
    'Exp. Readmissions Averted': round(r['expected_readmissions_averted'], 2),
    'True Readmissions Caught': r['actual_readmissions_in_selected'],
    'Cohort Precision %': round(r['precision_in_allocated'] * 100, 2)
} for r in allocation_results])

display(alloc_summary)

,Nurse Hour Budget,Patients Reached,Hours Utilized,Exp. Readmissions Averted,True Readmissions Caught,Cohort Precision %
0,500,176,499.8,21.35,64,36.36
1,1500,465,1498.0,52.48,131,28.17
2,3000,879,2997.5,91.15,224,25.48


**Cell 4: Marginal Returns Under Staffing Constraints**

Evaluate the point of diminishing marginal returns by sweeping staffing capacity from 200 to 5,000 nurse-hours.

In [4]:
sweep_budgets = np.linspace(200, 5000, 25)
sweep_records = []

for b in sweep_budgets:
    res = optimize_under_capacity(high_risk_cohort, b)
    sweep_records.append({
        'Nurse_Hours': b,
        'Patients_Contacted': res['allocated_patients'],
        'Readmissions_Averted': res['expected_readmissions_averted']
    })

sweep_df = pd.DataFrame(sweep_records)

# Calculate marginal yield: additional readmissions averted per 100 additional nurse-hours
sweep_df['marginal_yield_per_100h'] = (
    sweep_df['Readmissions_Averted'].diff() / sweep_df['Nurse_Hours'].diff()
) * 100

print("=== CAPACITY ALLOCATION FRONTIER ===")
display(sweep_df.dropna().head(10))

=== CAPACITY ALLOCATION FRONTIER ===


,Nurse_Hours,Patients_Contacted,Readmissions_Averted,marginal_yield_per_100h
1,400.0,144,17.726743,3.909777
2,600.0,206,24.832801,3.553029
3,800.0,269,31.465505,3.316352
4,1000.0,326,37.759646,3.147071
5,1200.0,387,43.851166,3.045760
6,1400.0,436,49.645093,2.896964
7,1600.0,499,55.337504,2.846206
8,1800.0,556,60.827991,2.745243
9,2000.0,609,66.185159,2.678584
10,2200.0,662,71.408075,2.611458


**Cell 5: Persist Simulation Artifacts**

Serialize the simulation data for inclusion in your research summary tables.

In [5]:
simulation_artifacts = {
    'allocation_summary': alloc_summary,
    'capacity_frontier': sweep_df,
    'monte_carlo_bounds': {
        'mean_averted': total_mean_averted,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }
}

sim_export_path = os.path.join(DATA_PROCESSED, 'simulation_artifacts.joblib')
joblib.dump(simulation_artifacts, sim_export_path)
print(f"Simulation artifacts persisted to: {sim_export_path}")

Simulation artifacts persisted to: /content/drive/MyDrive/DiaVigil/data/processed/simulation_artifacts.joblib
